<a href="https://colab.research.google.com/github/kxenopoulou/xenopoulos-logic-dialectic/blob/main/Dynamic_Xenopoulos_Dialectical_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ===================================================================
# INTERACTIVE XENOPOULOS SYSTEM WITH PARAMETER CONTROLS
# ===================================================================

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import time
import os
import json
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
import warnings
warnings.filterwarnings('ignore')

print("✅ Interactive Xenopoulos System Loaded")
print("📊 Use the controls below to adjust parameters in real-time!")

# ===================================================================
# CORE SYSTEM CLASSES
# ===================================================================

class InteractiveXenopoulosSystem:
    """Interactive system with real-time parameter controls"""

    def __init__(self):
        self.system = None
        self.results = None
        self.fig = None
        self.axs = None

        # Default parameters
        self.params = {
            'dimension': 3,
            'epochs': 200,
            'chaos_factor': 0.03,
            'quality_threshold': 0.8,
            'growth_rate': 1.2,
            'competition_strength': 0.4,
            'cooperation_factor': 0.1,
            'noise_intensity': 0.02,
            'alpha': 0.7,
            'beta': 0.3,
            'gamma': 0.4,
            'history_depth': 3
        }

        # Create widgets for all parameters
        self._create_widgets()

    def _create_widgets(self):
        """Create interactive widgets for all parameters"""

        # Basic parameters
        self.dim_widget = widgets.IntSlider(
            value=self.params['dimension'],
            min=2, max=10, step=1,
            description='Διάσταση:',
            style={'description_width': 'initial'},
            continuous_update=False
        )

        self.epochs_widget = widgets.IntSlider(
            value=self.params['epochs'],
            min=50, max=1000, step=50,
            description='Εποχές:',
            style={'description_width': 'initial'},
            continuous_update=False
        )

        self.chaos_widget = widgets.FloatSlider(
            value=self.params['chaos_factor'],
            min=0.0, max=0.2, step=0.01,
            description='Χάος:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.threshold_widget = widgets.FloatSlider(
            value=self.params['quality_threshold'],
            min=0.1, max=2.0, step=0.1,
            description='Όριο Ποιότητας:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        # Dialectical parameters
        self.alpha_widget = widgets.FloatSlider(
            value=self.params['alpha'],
            min=0.1, max=1.5, step=0.1,
            description='α (I•N weight):',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.beta_widget = widgets.FloatSlider(
            value=self.params['beta'],
            min=0.1, max=1.5, step=0.1,
            description='β (|I-N| weight):',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.gamma_widget = widgets.FloatSlider(
            value=self.params['gamma'],
            min=0.1, max=1.5, step=0.1,
            description='γ (R weight):',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        # Ontological conflict parameters
        self.growth_widget = widgets.FloatSlider(
            value=self.params['growth_rate'],
            min=0.5, max=3.0, step=0.1,
            description='Ρυθμός Ανάπτυξης:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.competition_widget = widgets.FloatSlider(
            value=self.params['competition_strength'],
            min=0.1, max=1.0, step=0.1,
            description='Δύναμη Ανταγωνισμού:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.cooperation_widget = widgets.FloatSlider(
            value=self.params['cooperation_factor'],
            min=0.0, max=0.5, step=0.05,
            description='Παράγοντας Συνεργασίας:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        self.noise_widget = widgets.FloatSlider(
            value=self.params['noise_intensity'],
            min=0.0, max=0.1, step=0.01,
            description='Εντάση Θορύβου:',
            style={'description_width': 'initial'},
            readout_format='.2f',
            continuous_update=False
        )

        # Control buttons
        self.run_button = widgets.Button(
            description='🚀 ΕΚΤΕΛΕΣΗ ΣΥΜΠΑΝΤΩΝ',
            button_style='success',
            layout=widgets.Layout(width='auto', height='40px')
        )

        self.reset_button = widgets.Button(
            description='🔄 ΕΠΑΝΑΦΟΡΑ',
            button_style='warning',
            layout=widgets.Layout(width='auto', height='40px')
        )

        self.save_button = widgets.Button(
            description='💾 ΑΠΟΘΗΚΕΥΣΗ',
            button_style='info',
            layout=widgets.Layout(width='auto', height='40px')
        )

        # Output area
        self.output = widgets.Output()

        # Connect button events
        self.run_button.on_click(self.run_simulation)
        self.reset_button.on_click(self.reset_parameters)
        self.save_button.on_click(self.save_results)

    def get_current_parameters(self):
        """Get current parameter values from widgets"""
        return {
            'dimension': self.dim_widget.value,
            'epochs': self.epochs_widget.value,
            'chaos_factor': self.chaos_widget.value,
            'quality_threshold': self.threshold_widget.value,
            'growth_rate': self.growth_widget.value,
            'competition_strength': self.competition_widget.value,
            'cooperation_factor': self.cooperation_widget.value,
            'noise_intensity': self.noise_widget.value,
            'alpha': self.alpha_widget.value,
            'beta': self.beta_widget.value,
            'gamma': self.gamma_widget.value
        }

    def reset_parameters(self, b):
        """Reset all parameters to defaults"""
        self.dim_widget.value = 3
        self.epochs_widget.value = 200
        self.chaos_widget.value = 0.03
        self.threshold_widget.value = 0.8
        self.growth_widget.value = 1.2
        self.competition_widget.value = 0.4
        self.cooperation_widget.value = 0.1
        self.noise_widget.value = 0.02
        self.alpha_widget.value = 0.7
        self.beta_widget.value = 0.3
        self.gamma_widget.value = 0.4

        with self.output:
            clear_output()
            print("✅ Παράμετροι επαναφέρθηκαν στις προεπιλογές")

    def create_system(self, params):
        """Create Xenopoulos system with given parameters"""

        class XenopoulosKlein4Group:
            def __init__(self, dim):
                self.dimension = dim
                self.I = np.eye(dim)
                self.N = -np.eye(dim)
                self.R = self._create_inverse_r(dim)
                self.C = self.N @ self.R

            def _create_inverse_r(self, dim):
                """Create INVERSE cyclic permutation"""
                R = np.zeros((dim, dim))
                for i in range(dim):
                    R[(i + 1) % dim, i] = 1.0
                return R

            def apply_operator(self, vector, operator):
                ops = {'I': self.I, 'N': self.N, 'R': self.R, 'C': self.C}
                return ops[operator] @ vector

        class DialecticalDynamics:
            def __init__(self, dim, alpha, beta, gamma, threshold):
                self.dimension = dim
                self.alpha = alpha
                self.beta = beta
                self.gamma = gamma
                self.threshold = threshold
                self.group = XenopoulosKlein4Group(dim)

            def synthesize(self, thesis, antithesis, mode='D1'):
                # Apply INRC operators
                I = thesis
                N = -antithesis
                R = self.group.apply_operator(thesis, 'R')
                C = self.group.apply_operator(thesis, 'C')

                # Xenopoulos synthesis equation
                I_dot_N = np.dot(I, N)
                I_minus_N_norm = np.linalg.norm(I - N)

                synthesis = (
                    self.alpha * I_dot_N -
                    self.beta * I_minus_N_norm +
                    self.gamma * np.mean(R)
                )

                # Simple combination
                if mode == 'D1':
                    final = 0.4 * I + 0.3 * N + 0.3 * R + 0.1 * synthesis
                else:
                    final = 0.3 * I + 0.4 * N + 0.2 * R + 0.1 * synthesis

                norm = np.linalg.norm(final)
                transition = norm > self.threshold

                return final, norm, transition, {'I': I, 'N': N, 'R': R, 'C': C}

        class OntologicalConflict:
            def __init__(self, dim, growth, competition, cooperation, noise):
                self.dimension = dim
                self.growth = growth
                self.competition = competition
                self.cooperation = cooperation
                self.noise = noise

            def evolve(self, thesis, antithesis, steps=10):
                # Simple conflict dynamics
                for _ in range(steps):
                    d_thesis = (
                        self.growth * thesis -
                        self.competition * thesis * antithesis +
                        self.cooperation * antithesis +
                        self.noise * np.random.randn(self.dimension)
                    )

                    d_antithesis = (
                        self.growth * antithesis -
                        self.competition * antithesis * thesis +
                        self.cooperation * thesis +
                        self.noise * np.random.randn(self.dimension)
                    )

                    thesis = thesis + 0.01 * d_thesis
                    antithesis = antithesis + 0.01 * d_antithesis

                return thesis, antithesis

        # Create the complete system
        system = type('System', (), {})()
        system.params = params
        system.group = XenopoulosKlein4Group(params['dimension'])
        system.dialectics = DialecticalDynamics(
            params['dimension'],
            params['alpha'],
            params['beta'],
            params['gamma'],
            params['quality_threshold']
        )
        system.ontology = OntologicalConflict(
            params['dimension'],
            params['growth_rate'],
            params['competition_strength'],
            params['cooperation_factor'],
            params['noise_intensity']
        )

        return system

    def run_simulation(self, b):
        """Run simulation with current parameters"""
        with self.output:
            clear_output(wait=True)

            # Get current parameters
            params = self.get_current_parameters()

            print("🎬 ΕΚΤΕΛΕΣΗ ΣΥΜΠΑΝΤΩΝ...")
            print("="*60)
            print("📋 ΠΑΡΑΜΕΤΡΟΙ ΣΥΣΤΗΜΑΤΟΣ:")
            print("-"*40)
            for key, value in params.items():
                print(f"  {key}: {value}")
            print("="*60)

            start_time = time.time()

            # Create system
            self.system = self.create_system(params)
            system = self.system

            # Initialize states
            dim = params['dimension']
            thesis = np.random.randn(dim)
            thesis = thesis / (np.linalg.norm(thesis) + 1e-8)
            antithesis = -0.8 * thesis + 0.2 * np.random.randn(dim)
            antithesis = antithesis / (np.linalg.norm(antithesis) + 1e-8)

            # Run simulation
            history = {
                'epochs': [],
                'thesis': [],
                'antithesis': [],
                'synthesis': [],
                'norms': [],
                'transitions': [],
                'operators': []
            }

            transitions = []

            for epoch in range(params['epochs']):
                # Dialectical synthesis
                mode = 'D1' if epoch % 2 == 0 else 'D2'
                synthesis, norm, transition, operators = system.dialectics.synthesize(
                    thesis, antithesis, mode
                )

                # Add chaos
                if params['chaos_factor'] > 0:
                    synthesis += params['chaos_factor'] * np.random.randn(dim)
                    norm = np.linalg.norm(synthesis)
                    if norm > 0:
                        synthesis = synthesis / norm

                # Evolve ontological conflict
                thesis, antithesis = system.ontology.evolve(thesis, antithesis)

                # Check for qualitative transition
                if transition:
                    transitions.append({
                        'epoch': epoch,
                        'norm': norm,
                        'old_thesis': thesis.copy(),
                        'new_thesis': synthesis.copy()
                    })

                    # Update states (negation of negation)
                    thesis = 0.6 * thesis + 0.4 * synthesis
                    thesis = thesis / (np.linalg.norm(thesis) + 1e-8)
                    antithesis = -0.7 * thesis + 0.3 * np.random.randn(dim)
                    antithesis = antithesis / (np.linalg.norm(antithesis) + 1e-8)

                # Store history
                history['epochs'].append(epoch)
                history['thesis'].append(thesis.copy())
                history['antithesis'].append(antithesis.copy())
                history['synthesis'].append(synthesis.copy())
                history['norms'].append(norm)
                history['transitions'].append(transition)
                history['operators'].append(operators)

                # Progress report
                if epoch % 50 == 0 and epoch > 0:
                    print(f"  [Εποχή {epoch}] Norm: {norm:.3f}, Μεταβάσεις: {len(transitions)}")

            elapsed_time = time.time() - start_time

            # Store results
            self.results = {
                'parameters': params,
                'history': history,
                'transitions': transitions,
                'group_matrices': {
                    'I': system.group.I.tolist(),
                    'N': system.group.N.tolist(),
                    'R': system.group.R.tolist(),
                    'C': system.group.C.tolist()
                },
                'statistics': self._calculate_statistics(history, transitions)
            }

            # Display results
            self._display_results()

            # Create visualizations
            self._create_visualizations(history, transitions)

            print(f"\n⏱️  Χρόνος εκτέλεσης: {elapsed_time:.2f} δευτερόλεπτα")
            print("✅ ΕΚΤΕΛΕΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ!")

    def _calculate_statistics(self, history, transitions):
        """Calculate system statistics"""
        norms = history['norms']

        stats = {
            'total_epochs': len(norms),
            'transition_count': len(transitions),
            'transition_rate': len(transitions) / len(norms) * 100,
            'norm_mean': np.mean(norms),
            'norm_max': np.max(norms),
            'norm_min': np.min(norms),
            'norm_std': np.std(norms),
            'operator_traces': {
                'I': np.trace(self.system.group.I),
                'N': np.trace(self.system.group.N),
                'R': np.trace(self.system.group.R),
                'C': np.trace(self.system.group.C)
            }
        }

        return stats

    def _display_results(self):
        """Display results in formatted output"""
        if not self.results:
            return

        stats = self.results['statistics']
        params = self.results['parameters']

        print("\n" + "="*70)
        print("ΑΠΟΤΕΛΕΣΜΑΤΑ ΣΥΜΠΑΝΤΩΝ")
        print("="*70)

        print("\n1. Σύστημα και Παράμετροι")
        print("-"*40)
        print(f"   Διάσταση (Dimension): {params['dimension']}")
        print(f"   Συνολικές εποχές (Total Epochs): {params['epochs']}")
        print(f"   Χάος (Chaos Factor): {params['chaos_factor']:.3f}")
        print(f"   Όριο ποιότητας (Quality Threshold): {params['quality_threshold']:.2f}")
        print(f"   Ρυθμός Ανάπτυξης (Growth Rate): {params['growth_rate']:.2f}")
        print(f"   Δύναμη Ανταγωνισμού (Competition): {params['competition_strength']:.2f}")
        print(f"   Συνεργασία (Cooperation): {params['cooperation_factor']:.2f}")

        print("\n2. Αποτελέσματα")
        print("-"*40)
        print(f"   Σύνολο Μεταβάσεων (Qualitative Transitions): {stats['transition_count']}")
        print(f"   Ρυθμός Μετάβασης (Transition Rate): {stats['transition_rate']:.1f}%")
        print(f"   Norms Σύνθεσης (Synthesis Norms):")
        print(f"     Μέσος όρος (Mean): {stats['norm_mean']:.3f}")
        print(f"     Μέγιστο (Maximum): {stats['norm_max']:.3f}")
        print(f"     Ελάχιστο (Minimum): {stats['norm_min']:.3f}")
        print(f"     Τυπική Απόκλιση (Std Dev): {stats['norm_std']:.3f}")

        print("\n3. Ίχνη Τελεστών INRC (Operator Traces)")
        print("-"*40)
        traces = stats['operator_traces']
        print(f"   I (Identity): {traces['I']:.2f}")
        print(f"   N (Negation): {traces['N']:.2f}")
        print(f"   R (Reciprocity - ΑΝΤΙΣΤΡΟΦΟΣ): {traces['R']:.2f}")
        print(f"   C (Correlation): {traces['C']:.2f}")

        # Check R properties
        R = self.system.group.R
        forward_R = np.zeros_like(R)
        dim = params['dimension']
        for i in range(dim):
            forward_R[i, (i + 1) % dim] = 1.0

        is_inverse = np.allclose(R @ forward_R, np.eye(dim))
        is_orthogonal = np.allclose(R @ R.T, np.eye(dim))

        print(f"\n4. Ιδιότητες Τελεστή R")
        print("-"*40)
        print(f"   Είναι αντίστροφος: {'✅' if is_inverse else '❌'}")
        print(f"   Είναι ορθογώνιος: {'✅' if is_orthogonal else '❌'}")
        print(f"   Ορίζουσα (Determinant): {np.linalg.det(R):.2f}")

    def _create_visualizations(self, history, transitions):
        """Create comprehensive visualizations"""
        print("\n🖼️  ΔΗΜΙΟΥΡΓΙΑ ΟΠΤΙΚΟΠΟΙΗΣΕΩΝ...")

        epochs = history['epochs']
        norms = history['norms']
        syntheses = np.array(history['synthesis'])

        # Create figure with GridSpec for better layout
        self.fig = plt.figure(figsize=(20, 12), facecolor='#f5f5f5')
        gs = GridSpec(3, 4, figure=self.fig, hspace=0.3, wspace=0.3)

        # 1. Synthesis Evolution
        ax1 = self.fig.add_subplot(gs[0, 0])
        ax1.plot(epochs, norms, 'b-', linewidth=2, alpha=0.8, label='Norm Σύνθεσης')
        ax1.axhline(self.threshold_widget.value, color='r', linestyle='--',
                   alpha=0.7, linewidth=1.5, label='Όριο Ποιότητας')

        if transitions:
            trans_epochs = [t['epoch'] for t in transitions]
            trans_norms = [t['norm'] for t in transitions]
            ax1.scatter(trans_epochs, trans_norms, color='gold', s=100,
                       edgecolors='black', zorder=5, label='Ποιότητικές Μεταβάσεις')

        ax1.set_title('Εξέλιξη Σύνθεσης', fontsize=14, fontweight='bold', pad=15)
        ax1.set_xlabel('Εποχή', fontsize=12)
        ax1.set_ylabel('Norm Σύνθεσης', fontsize=12)
        ax1.grid(True, alpha=0.3)
        ax1.legend(loc='best', fontsize=10)
        ax1.set_facecolor('#f9f9f9')

        # 2. Phase Space (2D Projection)
        ax2 = self.fig.add_subplot(gs[0, 1])
        if syntheses.shape[1] >= 2:
            colors = np.arange(len(syntheses))
            sc2 = ax2.scatter(syntheses[:, 0], syntheses[:, 1],
                            c=colors, cmap='viridis', s=30, alpha=0.7)
            ax2.plot(syntheses[:, 0], syntheses[:, 1], 'k-', alpha=0.2, linewidth=0.5)

            # Mark transitions
            if transitions:
                trans_points = syntheses[[t['epoch'] for t in transitions]]
                ax2.scatter(trans_points[:, 0], trans_points[:, 1],
                          color='red', s=80, marker='*', edgecolors='black',
                          label='Μεταβάσεις', zorder=5)

            ax2.set_title('Χώρος Φάσης (2Δ Προβολή)', fontsize=14, fontweight='bold', pad=15)
            ax2.set_xlabel('Συστατικό 1', fontsize=12)
            ax2.set_ylabel('Συστατικό 2', fontsize=12)
            ax2.grid(True, alpha=0.3)
            ax2.legend(loc='best', fontsize=10)
            ax2.set_facecolor('#f9f9f9')

            # Add colorbar
            cbar2 = plt.colorbar(sc2, ax=ax2)
            cbar2.set_label('Εποχή', fontsize=11)

        # 3. Norm Distribution
        ax3 = self.fig.add_subplot(gs[0, 2])
        n, bins, patches = ax3.hist(norms, bins=30, alpha=0.7, color='darkorange',
                                   edgecolor='black', density=True)

        # Add normal distribution curve
        from scipy.stats import norm as norm_dist
        mu, sigma = np.mean(norms), np.std(norms)
        x = np.linspace(min(norms), max(norms), 100)
        y = norm_dist.pdf(x, mu, sigma)
        ax3.plot(x, y, 'r-', linewidth=2, alpha=0.8, label='Κανονική Κατανομή')

        ax3.axvline(mu, color='blue', linestyle='--', alpha=0.7,
                   linewidth=1.5, label=f'Μέσος: {mu:.3f}')
        ax3.axvline(self.threshold_widget.value, color='green', linestyle=':',
                   alpha=0.7, linewidth=1.5, label=f'Όριο: {self.threshold_widget.value}')

        ax3.set_title('Κατανομή Norm Σύνθεσης', fontsize=14, fontweight='bold', pad=15)
        ax3.set_xlabel('Norm Σύνθεσης', fontsize=12)
        ax3.set_ylabel('Πυκνότητα', fontsize=12)
        ax3.grid(True, alpha=0.3)
        ax3.legend(loc='best', fontsize=10)
        ax3.set_facecolor('#f9f9f9')

        # 4. Autocorrelation
        ax4 = self.fig.add_subplot(gs[0, 3])
        if len(norms) > 50:
            autocorr = np.correlate(norms, norms, mode='full')
            autocorr = autocorr[len(norms)-1:] / autocorr[len(norms)-1]
            lags = range(min(50, len(autocorr)))

            ax4.plot(lags, autocorr[:len(lags)], 'k-', linewidth=2, alpha=0.8)
            ax4.axhline(0, color='r', linestyle='--', alpha=0.5, linewidth=1)

            # Highlight significant correlations
            sig_level = 2 / np.sqrt(len(norms))
            ax4.axhline(sig_level, color='green', linestyle=':', alpha=0.5, linewidth=1)
            ax4.axhline(-sig_level, color='green', linestyle=':', alpha=0.5, linewidth=1)
            ax4.fill_between(lags, -sig_level, sig_level, color='green', alpha=0.1)

            ax4.set_title('Αυτοσυσχέτιση Σύνθεσης', fontsize=14, fontweight='bold', pad=15)
            ax4.set_xlabel('Υστέρηση (Lag)', fontsize=12)
            ax4.set_ylabel('Συσχέτιση', fontsize=12)
            ax4.grid(True, alpha=0.3)
            ax4.set_facecolor('#f9f9f9')

        # 5. Operator Traces
        ax5 = self.fig.add_subplot(gs[1, 0])
        operators = ['I', 'N', 'R', 'C']
        traces = [
            np.trace(self.system.group.I),
            np.trace(self.system.group.N),
            np.trace(self.system.group.R),
            np.trace(self.system.group.C)
        ]

        colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
        bars = ax5.bar(operators, traces, color=colors, alpha=0.8, edgecolor='black')

        ax5.set_title('Ίχνη Τελεστών INRC', fontsize=14, fontweight='bold', pad=15)
        ax5.set_ylabel('Ίχνος (Trace)', fontsize=12)
        ax5.grid(True, alpha=0.3, axis='y')
        ax5.set_facecolor('#f9f9f9')

        # Add value labels
        for bar, trace in zip(bars, traces):
            height = bar.get_height()
            ax5.text(bar.get_x() + bar.get_width()/2., height,
                    f'{trace:.2f}', ha='center', va='bottom', fontsize=11)

        # 6. Transition Analysis
        ax6 = self.fig.add_subplot(gs[1, 1])
        if transitions:
            trans_epochs = [t['epoch'] for t in transitions]
            trans_norms = [t['norm'] for t in transitions]

            ax6.scatter(trans_epochs, trans_norms, s=100, c=trans_norms,
                       cmap='hot', edgecolors='black', zorder=5)

            # Add trend line
            if len(transitions) > 2:
                z = np.polyfit(trans_epochs, trans_norms, 1)
                p = np.poly1d(z)
                ax6.plot(trans_epochs, p(trans_epochs), 'b--', alpha=0.7, linewidth=1.5,
                        label=f'Τάση: {z[0]:.3f}x + {z[1]:.3f}')

            ax6.set_title('Ποιότητικές Μεταβάσεις', fontsize=14, fontweight='bold', pad=15)
            ax6.set_xlabel('Εποχή', fontsize=12)
            ax6.set_ylabel('Norm στη Μετάβαση', fontsize=12)
            ax6.grid(True, alpha=0.3)
            ax6.legend(loc='best', fontsize=10)
            ax6.set_facecolor('#f9f9f9')
        else:
            ax6.text(0.5, 0.5, 'ΔΕΝ ΒΡΕΘΗΚΑΝ\nΠΟΙΟΤΗΤΙΚΕΣ ΜΕΤΑΒΑΣΕΙΣ',
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    transform=ax6.transAxes, color='gray')
            ax6.set_facecolor('#f9f9f9')
            ax6.set_title('Ποιότητικές Μεταβάσεις', fontsize=14, fontweight='bold', pad=15)

        # 7. Operator Matrices (Heatmaps)
        ax7 = self.fig.add_subplot(gs[1, 2])
        matrices = [self.system.group.I, self.system.group.N,
                   self.system.group.R, self.system.group.C]
        titles = ['I: Identity', 'N: Negation', 'R: Reciprocity', 'C: Correlation']

        # Combine matrices for visualization
        combined = np.zeros((self.dim_widget.value * 2, self.dim_widget.value * 2))
        for i in range(2):
            for j in range(2):
                idx = i * 2 + j
                if idx < len(matrices):
                    start_row = i * self.dim_widget.value
                    start_col = j * self.dim_widget.value
                    combined[start_row:start_row+self.dim_widget.value,
                            start_col:start_col+self.dim_widget.value] = matrices[idx]

        im = ax7.imshow(combined, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax7.set_title('Πίνακες Τελεστών INRC', fontsize=14, fontweight='bold', pad=15)
        ax7.set_xticks([])
        ax7.set_yticks([])

        # Add subplot labels
        for i in range(2):
            for j in range(2):
                idx = i * 2 + j
                if idx < len(titles):
                    x = j * self.dim_widget.value + self.dim_widget.value / 2 - 0.5
                    y = i * self.dim_widget.value + self.dim_widget.value / 2 - 0.5
                    ax7.text(x, y - self.dim_widget.value/2 - 0.3,
                            titles[idx], ha='center', va='center',
                            fontsize=10, fontweight='bold',
                            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        plt.colorbar(im, ax=ax7, label='Τιμή Πίνακα')
        ax7.set_facecolor('#f9f9f9')

        # 8. System Statistics
        ax8 = self.fig.add_subplot(gs[1, 3])
        ax8.axis('off')

        stats = self.results['statistics']
        params = self.results['parameters']

        info_text = f"""
        ΣΤΑΤΙΣΤΙΚΑ ΣΥΣΤΗΜΑΤΟΣ
        {'='*40}

        ΠΑΡΑΜΕΤΡΟΙ:
        Διάσταση: {params['dimension']}
        Εποχές: {params['epochs']}
        Χάος: {params['chaos_factor']:.3f}
        Όριο: {params['quality_threshold']:.2f}

        ΑΠΟΤΕΛΕΣΜΑΤΑ:
        Μεταβάσεις: {stats['transition_count']}
        Ρυθμός: {stats['transition_rate']:.1f}%

        NORMS:
        Μέσος: {stats['norm_mean']:.3f}
        Max: {stats['norm_max']:.3f}
        Min: {stats['norm_min']:.3f}
        Std: {stats['norm_std']:.3f}

        ΤΕΛΕΣΤΗΣ R:
        Ίχνος: {stats['operator_traces']['R']:.2f}
        Ορίζουσα: {np.linalg.det(self.system.group.R):.2f}
        Αντίστροφος: {'ΝΑΙ' if np.allclose(self.system.group.R @
                                          self.system.group.R.T,
                                          np.eye(params['dimension'])) else 'ΟΧΙ'}

        {'='*40}
        Δημιουργήθηκε: {time.strftime('%Y-%m-%d %H:%M:%S')}
        """

        ax8.text(0.05, 0.5, info_text, fontsize=10, family='monospace',
                verticalalignment='center', transform=ax8.transAxes,
                bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

        # 9. Time Series Analysis (Bottom row, full width)
        ax9 = self.fig.add_subplot(gs[2, :])

        if syntheses.shape[1] >= 3:
            # Plot first 3 components
            for i in range(min(3, syntheses.shape[1])):
                ax9.plot(epochs, syntheses[:, i],
                        label=f'Συστατικό {i+1}', alpha=0.8, linewidth=1.5)

            # Highlight transitions
            if transitions:
                for t in transitions:
                    ax9.axvline(t['epoch'], color='red', alpha=0.3, linestyle='--')

            ax9.set_title('Χρονικές Σειρές Συστατικών', fontsize=14, fontweight='bold', pad=15)
            ax9.set_xlabel('Εποχή', fontsize=12)
            ax9.set_ylabel('Τιμή Συστατικού', fontsize=12)
            ax9.grid(True, alpha=0.3)
            ax9.legend(loc='best', fontsize=10, ncol=3)
            ax9.set_facecolor('#f9f9f9')

        # Main title
        self.fig.suptitle(
            f'ΣΥΣΤΗΜΑ ΧΕΝΟΠΟΥΛΟΥ - ΔΙΑΛΕΚΤΙΚΗ ΔΥΝΑΜΙΚΗ\n'
            f'Διάσταση: {params["dimension"]} | Χάος: {params["chaos_factor"]:.3f} | '
            f'Όριο: {params["quality_threshold"]:.2f}',
            fontsize=16, fontweight='bold', y=1.02
        )

        plt.tight_layout()
        plt.show()

        print("✅ Οπτικοποιήσεις δημιουργήθηκαν επιτυχώς!")

    def save_results(self, b):
        """Save all results to files"""
        if not self.results:
            with self.output:
                print("❌ Δεν υπάρχουν αποτελέσματα για αποθήκευση")
            return

        timestamp = time.strftime("%Y%m%d_%H%M%S")
        folder = "xenopoulos_results"
        os.makedirs(folder, exist_ok=True)

        base_name = f"{folder}/xenopoulos_{timestamp}"

        # Save data
        np.save(f"{base_name}.npy", self.results)

        # Save JSON
        with open(f"{base_name}.json", 'w', encoding='utf-8') as f:
            json.dump(self.results, f, indent=2, ensure_ascii=False)

        # Save figure
        if self.fig:
            self.fig.savefig(f"{base_name}.png", dpi=300, bbox_inches='tight')
            self.fig.savefig(f"{base_name}.pdf", bbox_inches='tight')

        # Create HTML report
        html_content = f"""
        <!DOCTYPE html>
        <html>
        <head>
            <title>Xenopoulos System Results</title>
            <style>
                body {{ font-family: Arial, sans-serif; margin: 40px; }}
                .header {{ background: #2c3e50; color: white; padding: 20px; border-radius: 10px; }}
                .section {{ background: #f8f9fa; padding: 20px; margin: 20px 0; border-radius: 10px; }}
                .param {{ background: #e9ecef; padding: 10px; margin: 5px; border-radius: 5px; }}
                .stats {{ background: #d4edda; padding: 15px; border-radius: 5px; }}
            </style>
        </head>
        <body>
            <div class="header">
                <h1>Χενόπουλος Διαλεκτικό Σύστημα - Αποτελέσματα</h1>
                <p>Δημιουργήθηκε: {time.strftime('%Y-%m-%d %H:%M:%S')}</p>
            </div>

            <div class="section">
                <h2>Παράμετροι Συστήματος</h2>
                {''.join(f'<div class="param"><strong>{k}:</strong> {v}</div>'
                        for k, v in self.results['parameters'].items())}
            </div>

            <div class="section">
                <h2>Στατιστικά</h2>
                <div class="stats">
                    <p><strong>Μεταβάσεις:</strong> {self.results['statistics']['transition_count']}</p>
                    <p><strong>Μέσο Norm:</strong> {self.results['statistics']['norm_mean']:.3f}</p>
                    <p><strong>Τυπική Απόκλιση:</strong> {self.results['statistics']['norm_std']:.3f}</p>
                </div>
            </div>

            <div class="section">
                <h2>Αρχεία</h2>
                <ul>
                    <li><a href="{base_name}.npy">NumPy αρχείο (.npy)</a></li>
                    <li><a href="{base_name}.json">JSON αρχείο (.json)</a></li>
                    <li><a href="{base_name}.png">Εικόνα αποτελεσμάτων (.png)</a></li>
                    <li><a href="{base_name}.pdf">PDF αναφορά (.pdf)</a></li>
                </ul>
            </div>
        </body>
        </html>
        """

        with open(f"{base_name}.html", 'w', encoding='utf-8') as f:
            f.write(html_content)

        with self.output:
            clear_output()
            print("💾 ΑΠΟΘΗΚΕΥΣΗ ΟΛΟΚΛΗΡΩΘΗΚΕ!")
            print("="*60)
            print("📁 ΑΡΧΕΙΑ ΠΟΥ ΑΠΟΘΗΚΕΥΤΗΚΑΝ:")
            print(f"   • {base_name}.npy - Δεδομένα NumPy")
            print(f"   • {base_name}.json - Δεδομένα JSON")
            print(f"   • {base_name}.png - Εικόνα αποτελεσμάτων")
            print(f"   • {base_name}.pdf - PDF αναφορά")
            print(f"   • {base_name}.html - HTML αναφορά")
            print("="*60)
            print("✅ Όλα τα αρχεία αποθηκεύτηκαν στον φάκελο 'xenopoulos_results/'")

    def display_control_panel(self):
        """Display the complete control panel"""
        print("\n" + "="*70)
        print("🎛️  ΠΑΝΕΛΟ ΧΕΝΤΙΣΤΗΡΙΩΝ ΧΕΝΟΠΟΥΛΟΥ ΣΥΣΤΗΜΑΤΟΣ")
        print("="*70)

        # Create tabs for better organization
        tab1 = widgets.VBox([
            widgets.HTML("<h3 style='color: #2E86AB;'>ΒΑΣΙΚΕΣ ΠΑΡΑΜΕΤΡΟΙ</h3>"),
            self.dim_widget,
            self.epochs_widget,
            self.chaos_widget,
            self.threshold_widget,
            widgets.HTML("<hr style='border: 1px solid #ddd;'>")
        ])

        tab2 = widgets.VBox([
            widgets.HTML("<h3 style='color: #A23B72;'>ΔΙΑΛΕΚΤΙΚΕΣ ΠΑΡΑΜΕΤΡΟΙ</h3>"),
            self.alpha_widget,
            self.beta_widget,
            self.gamma_widget,
            widgets.HTML("<hr style='border: 1px solid #ddd;'>")
        ])

        tab3 = widgets.VBox([
            widgets.HTML("<h3 style='color: #F18F01;'>ΟΝΤΟΛΟΓΙΚΗ ΣΥΓΚΡΟΥΣΗ</h3>"),
            self.growth_widget,
            self.competition_widget,
            self.cooperation_widget,
            self.noise_widget,
            widgets.HTML("<hr style='border: 1px solid #ddd;'>")
        ])

        # Create tabs
        tabs = widgets.Tab([tab1, tab2, tab3])
        tabs.set_title(0, 'Βασικές')
        tabs.set_title(1, 'Διαλεκτικές')
        tabs.set_title(2, 'Σύγκρουση')

        # Control buttons
        buttons = widgets.HBox([
            self.run_button,
            self.reset_button,
            self.save_button
        ], layout=widgets.Layout(
            justify_content='center',
            margin='20px 0'
        ))

        # Display everything
        display(widgets.VBox([
            widgets.HTML("<h2 style='text-align: center; color: #2c3e50;'>"
                        "ΔΙΑΔΡΑΣΤΙΚΟ ΣΥΣΤΗΜΑ ΧΕΝΟΠΟΥΛΟΥ</h2>"),
            widgets.HTML("<p style='text-align: center; color: #666;'>"
                        "Ρυθμίστε τις παραμέτρους και πατήστε ΕΚΤΕΛΕΣΗ</p>"),
            tabs,
            buttons,
            self.output
        ]))

# ===================================================================
# CREATE AND RUN THE INTERACTIVE SYSTEM
# ===================================================================

# Create the interactive system
interactive_system = InteractiveXenopoulosSystem()

# Display the control panel
interactive_system.display_control_panel()

print("\n" + "="*70)
print("ΟΔΗΓΙΕΣ ΧΡΗΣΗΣ:")
print("="*70)
print("""
1. ΡΥΘΜΙΣΤΕ ΤΙΣ ΠΑΡΑΜΕΤΡΟΥΣ:
   • Χρησιμοποιήστε τα sliders για να αλλάξετε τιμές
   • Οι καρτέλες οργανώνουν τις παραμέτρους

2. ΕΚΤΕΛΕΣΤΕ ΤΟ ΣΥΣΤΗΜΑ:
   • Πατήστε το πράσινο κουμπί "ΕΚΤΕΛΕΣΗ ΣΥΜΠΑΝΤΩΝ"
   • Το σύστημα θα τρέξει με τις τρέχουσες παραμέτρους
   • Θα δείτε αναλυτικά αποτελέσματα και διαγράμματα

3. ΑΠΟΘΗΚΕΥΣΤΕ ΤΑ ΑΠΟΤΕΛΕΣΜΑΤΑ:
   • Πατήστε το μπλε κουμπί "ΑΠΟΘΗΚΕΥΣΗ"
   • Όλα τα δεδομένα και διαγράμματα θα αποθηκευτούν
   • Θα δημιουργηθεί HTML αναφορά

4. ΕΠΑΝΑΦΕΡΤΕ ΤΙΣ ΠΡΟΕΠΙΛΟΓΕΣ:
   • Πατήστε το κίτρινο κουμπί "ΕΠΑΝΑΦΟΡΑ"

📊 Ο ΤΕΛΕΣΤΗΣ R ΕΙΝΑΙ ΠΑΝΤΑ ΑΝΤΙΣΤΡΟΦΟΣ!
   - R[(i+1)%n, i] = 1.0 (αντίστροφη κυκλική μετάθεση)
   - C = N ∘ R = R ∘ N
   - Όλες οι ιδιότητες της ομάδας Klein-4 διατηρούνται
""")

# Run a quick demo automatically
print("\n🎯 Για να ξεκινήσετε αμέσως, πατήστε το πράσινο κουμπί ΕΚΤΕΛΕΣΗ!")
print("="*70)

✅ Interactive Xenopoulos System Loaded
📊 Use the controls below to adjust parameters in real-time!

🎛️  ΠΑΝΕΛΟ ΧΕΝΤΙΣΤΗΡΙΩΝ ΧΕΝΟΠΟΥΛΟΥ ΣΥΣΤΗΜΑΤΟΣ



ΟΔΗΓΙΕΣ ΧΡΗΣΗΣ:

1. ΡΥΘΜΙΣΤΕ ΤΙΣ ΠΑΡΑΜΕΤΡΟΥΣ:
   • Χρησιμοποιήστε τα sliders για να αλλάξετε τιμές
   • Οι καρτέλες οργανώνουν τις παραμέτρους

2. ΕΚΤΕΛΕΣΤΕ ΤΟ ΣΥΣΤΗΜΑ:
   • Πατήστε το πράσινο κουμπί "ΕΚΤΕΛΕΣΗ ΣΥΜΠΑΝΤΩΝ"
   • Το σύστημα θα τρέξει με τις τρέχουσες παραμέτρους
   • Θα δείτε αναλυτικά αποτελέσματα και διαγράμματα

3. ΑΠΟΘΗΚΕΥΣΤΕ ΤΑ ΑΠΟΤΕΛΕΣΜΑΤΑ:
   • Πατήστε το μπλε κουμπί "ΑΠΟΘΗΚΕΥΣΗ"
   • Όλα τα δεδομένα και διαγράμματα θα αποθηκευτούν
   • Θα δημιουργηθεί HTML αναφορά

4. ΕΠΑΝΑΦΕΡΤΕ ΤΙΣ ΠΡΟΕΠΙΛΟΓΕΣ:
   • Πατήστε το κίτρινο κουμπί "ΕΠΑΝΑΦΟΡΑ"

📊 Ο ΤΕΛΕΣΤΗΣ R ΕΙΝΑΙ ΠΑΝΤΑ ΑΝΤΙΣΤΡΟΦΟΣ!
   - R[(i+1)%n, i] = 1.0 (αντίστροφη κυκλική μετάθεση)
   - C = N ∘ R = R ∘ N
   - Όλες οι ιδιότητες της ομάδας Klein-4 διατηρούνται


🎯 Για να ξεκινήσετε αμέσως, πατήστε το πράσινο κουμπί ΕΚΤΕΛΕΣΗ!
